In [1]:
from ggset import GGSet
import os
from pathlib import Path
from zipfile import ZipFile
import shutil


def print_folder_structure(path: Path, depth: int = 0) -> None:
    for item in path.iterdir():
        if item.is_dir():
            print("  " * depth + item.name + "/")
            print_folder_structure(item, depth + 1)
        else:
            print("  " * depth + item.name)
            

# CSV Collection Dataset

When you have a dataset, where the labels are in a CSV file, you can use a `GGBulkCsvFileCollection` to read the labels.

In [2]:
dataset_path = Path("example/csv_collection_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

csv_coll_set = GGSet(dataset_path)

# get the train labels
with csv_coll_set.get_sub_dir("train").create_bulk_csv_collection("labels.csv", layer=2, rel_paths=True, caching=True) as labels:
    print("Iterating over train labels:")
    for file, label in labels:
        img = file.read_image()
        print(f"File: {file.rel_path}, Average: {img.mean():.2f}, Label: {label}")


# get all labels 
with csv_coll_set.create_bulk_csv_collection("labels.csv", layer=2, rel_paths=True) as labels:
    df = labels.read_dataframe()
    print("\nAll labels:")
    print(df)

# write new labels
with csv_coll_set.create_bulk_csv_collection("labels2.csv", layer=2, rel_paths=True, caching=False) as labels2:
    # Warning: As caching is disabled, any data will just be appended to the CSV file, so if you run this multiple times, you will get duplicate entries.
    # To avoid this, set caching=True, but then, the while file will be held in memory as a pandas DataFrame.
    # Alternatively, use csv_coll_set.get_existing_files_set() to get a set of files that already have labels, and skip those.
    for file in csv_coll_set.iterate(filter_endings=(".png",)):
        img = file.read_image()
        avg = img.mean()
        std = img.std()
        labels2.write(file, {"average": avg, "std": std})

with csv_coll_set.create_bulk_csv_collection("labels2.csv", layer=2, rel_paths=True) as labels2:
    df = labels2.read_dataframe()
    # When some label2.csv files or corresponding image files are missing, the corresponding rows will be missing in the DataFrame.
    # When one file has different columns then another file, the missing columns in the resulting DataFrame will be filled with NaN values.
    print("\nAll labels2:")
    print(df)

Folder structure of the extracted dataset:
test/
  labels.csv
  1020.png
  1010.png
train/
  labels.csv
  100.png
  10.png
  some_extra_numbers/
    1004.png
    1003.png
  1.png



Iterating over train labels:
File: train/100.png, Average: 23.18, Label: {'number': 6}
File: train/10.png, Average: 37.96, Label: {'number': 0}
File: train/1.png, Average: 36.80, Label: {'number': 2}
File: train/some_extra_numbers/1004.png, Average: 17.25, Label: {'number': 1}
File: train/some_extra_numbers/1003.png, Average: 58.40, Label: {'number': 5}

All labels:
                            filename  number
0                      test/1020.png       3
1                      test/1010.png       4
2                      train/100.png       6
3                       train/10.png       0
4                        train/1.png       2
5  train/some_extra_numbers/1004.png       1
6  train/some_extra_numbers/1003.png       5

All labels2:
                            filename    average         std
0              

# Json collection dataset

When you have a dataset, where the labels are in a JSON file, you can use a `GGBulkJsonFileCollection` to read the labels.

In [3]:
dataset_path = Path("example/json_collection_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

json_coll_set = GGSet(dataset_path)

# get the train labels
with json_coll_set.get_sub_dir("train").create_bulk_json_collection("labels.json", layer=2, rel_paths=True) as labels:
    print("Iterating over train labels:")
    for file, label in labels:
        img = file.read_image()
        print(f"File: {file.rel_path}, Average: {img.mean():.2f}, Label: {label}")

# get all labels
with json_coll_set.create_bulk_json_collection("labels.json", layer=2, rel_paths=True) as labels:
    data = labels.read_dict()
    print("\nAll labels:")
    print(df)

# write new labels
with json_coll_set.create_bulk_json_collection("labels2.json", layer=2, rel_paths=True) as labels2:
    for file in json_coll_set.iterate(filter_endings=(".png",)):
        img = file.read_image()
        avg = img.mean()
        std = img.std()
        labels2.write(file, {"average": avg, "std": std})

print_folder_structure(dataset_path)




Folder structure of the extracted dataset:
test/
  labels.json
  1020.png
  1010.png
train/
  labels.json
  100.png
  10.png
  some_extra_numbers/
    1004.png
    1003.png
  1.png



Iterating over train labels:
File: train/100.png, Average: 23.18, Label: {'number': 6}
File: train/10.png, Average: 37.96, Label: {'number': 0}
File: train/1.png, Average: 36.80, Label: {'number': 2}
File: train/some_extra_numbers/1004.png, Average: 17.25, Label: {'number': 1}
File: train/some_extra_numbers/1003.png, Average: 58.40, Label: {'number': 5}

All labels:
                            filename    average         std
0                      test/1020.png  25.631378   68.274809
1                      test/1010.png  40.568878   86.289190
2                      train/100.png  23.176020   61.125620
3                       train/10.png  37.960459   81.635543
4                        train/1.png  36.798469   81.885623
5  train/some_extra_numbers/1004.png  17.248724   58.692889
6  train/some_extra_numbers

# Branched datasets

When you have a dataset, where there is one directory for data and in another branch directory there are labels, you can set the `data_type_level` to the corresponding level of the directory structure. 


In [4]:
dataset_path = Path("example/branched_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

branched_set = GGSet(dataset_path, data_type_level=2)

# reading images and their corresponding labels from a branched dataset
for file in branched_set.get_sub_dir("train").iterate(data_type="images", filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    label_file = file.get_corresponding_file(data_type="labels", extension=".json")
    assert label_file is not None, f"No corresponding label file found for {file.rel_path}"
    label_data = label_file.read_json()
    print(f"File: {file.rel_path}, Average: {avg:.2f}, Label: {label_data} from {label_file.rel_path}")

print("\n\n")

# writing new labels to a branched dataset
for file in branched_set.iterate(data_type="images", filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    std = img.std()
    label_file = file.get_corresponding_file(data_type="labels2", extension=".yaml")
    assert label_file is not None, f"Could not create corresponding label file for {file.rel_path}"
    label_file.write_yaml({"average": float(avg), "std": float(std)})

for file in branched_set.iterate(data_type="images", filter_endings=(".png",)):
    label_file = file.get_corresponding_file(data_type="labels2", extension=".yaml")
    assert label_file is not None, f"No corresponding label file found for {file.rel_path}"
    label_data = label_file.read_yaml()
    print(f"File: {file.rel_path}, Label2: {label_data} from {label_file.rel_path}")



Folder structure of the extracted dataset:
test/
  images/
    1020.png
    1010.png
  labels/
    1020.json
    1010.json
train/
  images/
    100.png
    10.png
    some_extra_numbers/
      1004.png
      1003.png
    1.png
  labels/
    10.json
    1.json
    some_extra_numbers/
      1004.json
      1003.json
    100.json



File: train/images/100.png, Average: 23.18, Label: {'number': 6} from train/labels/100.json
File: train/images/10.png, Average: 37.96, Label: {'number': 0} from train/labels/10.json
File: train/images/1.png, Average: 36.80, Label: {'number': 2} from train/labels/1.json
File: train/images/some_extra_numbers/1004.png, Average: 17.25, Label: {'number': 1} from train/labels/some_extra_numbers/1004.json
File: train/images/some_extra_numbers/1003.png, Average: 58.40, Label: {'number': 5} from train/labels/some_extra_numbers/1003.json



File: test/images/1020.png, Label2: {'average': 25.631377551020407, 'std': 68.27480923938039} from test/labels2/1020.yaml
File: tes

# Same directory labels datasets

When you have a dataset, where there is one directory with all data and labels are in the same directory, you shall not the `type_sep_level` parameter. It defaults to `-1`, which means there are no separate directories for data and labels.

In [5]:
dataset_path = Path("example/same_dir_labels_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

same_dir_labels_set = GGSet(dataset_path)

# reading images and their corresponding labels from a dataset with same directory labels
for file in same_dir_labels_set.get_sub_dir("train").iterate(filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    label_file = file.get_corresponding_file_in_same_dir(".json")
    assert label_file is not None, f"No corresponding label file found for {file.rel_path}"
    label_data = label_file.read_json()
    print(f"File: {file.rel_path}, Average: {avg:.2f}, Label: {label_data} from {label_file.rel_path}")

# writing new labels to a dataset with same directory labels
for file in same_dir_labels_set.iterate(filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    std = img.std()
    label_file = file.get_corresponding_file_in_same_dir(".yaml")
    assert label_file is not None, f"Could not create corresponding label file for {file.rel_path}"
    label_file.write_yaml({"average": float(avg), "std": float(std)})

print("\n\n")

for file in same_dir_labels_set.iterate(filter_endings=(".png",)):
    label_file = file.get_corresponding_file_in_same_dir(".yaml")
    assert label_file is not None, f"No corresponding label file found for {file.rel_path}"
    label_data = label_file.read_yaml()
    print(f"File: {file.rel_path}, Label2: {label_data} from {label_file.rel_path}")

Folder structure of the extracted dataset:
test/
  images/
    1020.png
    1020.json
    1010.png
    1010.json
train/
  images/
    10.json
    100.png
    1.json
    10.png
    some_extra_numbers/
      1004.json
      1003.json
      1004.png
      1003.png
    1.png
    100.json



File: train/images/100.png, Average: 23.18, Label: {'number': 6} from train/images/100.json
File: train/images/10.png, Average: 37.96, Label: {'number': 0} from train/images/10.json
File: train/images/1.png, Average: 36.80, Label: {'number': 2} from train/images/1.json
File: train/images/some_extra_numbers/1004.png, Average: 17.25, Label: {'number': 1} from train/images/some_extra_numbers/1004.json
File: train/images/some_extra_numbers/1003.png, Average: 58.40, Label: {'number': 5} from train/images/some_extra_numbers/1003.json



File: test/images/1020.png, Label2: {'average': 25.631377551020407, 'std': 68.27480923938039} from test/images/1020.yaml
File: test/images/1010.png, Label2: {'average': 40.568

# Datasets with labels in directories in the same level as the data

When you have a dataset, where there is one directory for data files and for each data file there is a corresponding directory with the same name as the data file, you can work with this dataset as follows:

In [6]:
dataset_path = Path("example/labels_in_dir_in_same_dir_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

labels_in_dir_in_same_dir_set = GGSet(dataset_path)

# read labels from a dataset with labels in directories in the same level as the data
for file in labels_in_dir_in_same_dir_set.iterate(filter_endings=(".png",)):
    img = file.read_image()
    label_dir = file.get_corresponding_dir_in_same_dir()
    assert label_dir is not None, f"No corresponding label directory found for {file.rel_path}"
    for label_file in label_dir.iterate(filter_endings=(".txt",)):
        label_data = label_file.read_text()
        print(f"File: {file.rel_path}, Label: '{label_data}' from {label_file.rel_path}")


# write new labels to a dataset with labels in directories in the same level as the data
for file in labels_in_dir_in_same_dir_set.iterate(filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    std = img.std()
    label_dir = file.get_corresponding_dir_in_same_dir()
    new_file_auto_named = label_dir.get_new_sub_file(".txt")
    new_file_auto_named.write_text(f"Average: {avg:.2f}, Std: {std:.2f}")
    new_file_named = label_dir.get_file("label_info.txt")
    new_file_named.write_text(f"A cool label for {file.rel_path}")

print_folder_structure(dataset_path)


Folder structure of the extracted dataset:
test/
  1020.png
  1010/
    2.txt
    1.txt
  1020/
    2.txt
    1.txt
  1010.png
train/
  100.png
  1/
    2.txt
    1.txt
  10/
    2.txt
    1.txt
  10.png
  some_extra_numbers/
    1003/
      2.txt
      1.txt
    1004/
      2.txt
      1.txt
    1004.png
    1003.png
  100/
    2.txt
    1.txt
  1.png



File: test/1020.png, Label: 'Std: 68.27' from test/1020/2.txt
File: test/1020.png, Label: 'Average: 25.63' from test/1020/1.txt
File: test/1010.png, Label: 'Std: 86.29' from test/1010/2.txt
File: test/1010.png, Label: 'Average: 40.57' from test/1010/1.txt
File: train/100.png, Label: 'Std: 61.13' from train/100/2.txt
File: train/100.png, Label: 'Average: 23.18' from train/100/1.txt
File: train/10.png, Label: 'Std: 81.64' from train/10/2.txt
File: train/10.png, Label: 'Average: 37.96' from train/10/1.txt
File: train/1.png, Label: 'Std: 81.89' from train/1/2.txt
File: train/1.png, Label: 'Average: 36.80' from train/1/1.txt
File: train/so

# Datasets with labels in directories in a different branch

When you have a dataset, where there is one directory for data files and for each data file there is a corresponding directory with the same name as the data file, but in a different branch of the directory structure, you can work with this dataset as follows:

In [7]:
dataset_path = Path("example/labels_in_dir_in_branch_dataset")
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)
with ZipFile(dataset_path.with_suffix(".zip"), 'r') as zip_ref:
    zip_ref.extractall(".")

print("Folder structure of the extracted dataset:")
print_folder_structure(dataset_path)
print("\n\n")

labels_in_dir_in_branch_set = GGSet(dataset_path, data_type_level=2)

# read labels from a dataset with labels in directories in the same level as the data
for file in labels_in_dir_in_branch_set.iterate(data_type="images", filter_endings=(".png",)):
    img = file.read_image()
    label_dir = file.get_corresponding_dir("labels")
    assert label_dir is not None, f"No corresponding label directory found for {file.rel_path}"
    for label_file in label_dir.iterate(filter_endings=(".txt",)):
        label_data = label_file.read_text()
        print(f"File: {file.rel_path}, Label: '{label_data}' from {label_file.rel_path}")


# write new labels to a dataset with labels in directories in the same level as the data
for file in labels_in_dir_in_branch_set.iterate(data_type="images", filter_endings=(".png",)):
    img = file.read_image()
    avg = img.mean()
    std = img.std()
    label_dir = file.get_corresponding_dir("labels")
    new_file_auto_named = label_dir.get_new_sub_file(".txt")
    new_file_auto_named.write_text(f"Average: {avg:.2f}, Std: {std:.2f}")
    new_file_named = label_dir.get_file("label_info.txt")
    new_file_named.write_text(f"A cool label for {file.rel_path}")

print_folder_structure(dataset_path)


Folder structure of the extracted dataset:
test/
  images/
    1020.png
    1010.png
  labels/
    1010/
      2.txt
      1.txt
    1020/
      2.txt
      1.txt
train/
  images/
    100.png
    10.png
    some_extra_numbers/
      1004.png
      1003.png
    1.png
  labels/
    1/
      2.txt
      1.txt
    10/
      2.txt
      1.txt
    some_extra_numbers/
      1003/
        2.txt
        1.txt
      1004/
        2.txt
        1.txt
    100/
      3.txt
      2.txt
      1.txt
      label_info.txt



File: test/images/1020.png, Label: 'Std: 68.27' from test/labels/1020/2.txt
File: test/images/1020.png, Label: 'Average: 25.63' from test/labels/1020/1.txt
File: test/images/1010.png, Label: 'Std: 86.29' from test/labels/1010/2.txt
File: test/images/1010.png, Label: 'Average: 40.57' from test/labels/1010/1.txt
File: train/images/100.png, Label: 'Average: 23.18, Std: 61.13' from train/labels/100/3.txt
File: train/images/100.png, Label: 'Std: 61.13' from train/labels/100/2.txt
File: t